# QLoRA fine-tune: Qwen2.5-1.5B-Instruct for customer support

Run top to bottom on a free T4 (v2 headline run target) or on Colab Pro (A100 or L4).
Nothing is mounted; every artifact is written under `train/runs/<run>/` and zipped at the end.

Order: GPU check, clone, install pinned versions, tokens, data, configuration, smoke run with a
resume proof, full run, per-checkpoint dev answers, curves, a selected-checkpoint cross-check
(merge plus optional Ollama serve), download.

Configs: `configs/train.yaml` (full grouped train split, Colab Pro) and `configs/train-t4.yaml`
(8,000 rows, the free-T4 budget; the served v1 checkpoint was trained on this config). Hyperparameters
are fixed by `docs/dag/CONTRACTS.md` section 5. This notebook has not been executed on Colab before
v2; read `notebooks/README.md` before running it for real.

## 1. GPU


In [ ]:
!nvidia-smi


## 2. Clone the repository

The repo is private. Add `GITHUB_TOKEN` to Colab secrets (a fine-grained PAT with read access to
`Imsharad/ghl-support-slm` is enough) before running this cell. The token is used to build the
clone URL in memory and is never printed. `BRANCH` defaults to `v2`; set it to a tag to reproduce
a specific submission.

**If this fails:** "could not read Username" or a 403 means the secret is missing, expired, or
lacks repo access; "Repository not found" with a correct token usually means the wrong branch name.

In [ ]:
import os

REPO = 'Imsharad/ghl-support-slm'
BRANCH = 'v2'
WORKDIR = '/content/ghl-support-slm'

try:
    from google.colab import userdata
    _token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    _token = None
    print('no GITHUB_TOKEN secret:', exc)

if not os.path.isdir(WORKDIR):
    if _token:
        _url = f'https://{_token}@github.com/{REPO}.git'
        get_ipython().system('git clone --branch $BRANCH --depth 1 -q $_url $WORKDIR')
    else:
        print('no token; trying an unauthenticated clone (works only if the repo is public)')
        get_ipython().system('git clone --branch $BRANCH --depth 1 -q https://github.com/$REPO.git $WORKDIR')
del _token  # never left resident longer than this cell needs it
os.chdir(WORKDIR)
!git rev-parse HEAD

## 3. Install the pinned versions

Colab ships different builds; these are the versions in `configs/versions.json` and `uv.lock`.
Restarting the runtime after this cell is normal if Colab asks.


In [ ]:
!pip install -q \
  'transformers==5.16.1' \
  'peft==0.20.0' \
  'trl==1.12.0' \
  'bitsandbytes==0.50.2' \
  'accelerate>=0.34' \
  'datasets==5.0.1' \
  'sentence-transformers>=3.0' \
  'scikit-learn>=1.5' \
  'pyyaml>=6.0' \
  'matplotlib>=3.9'

import torch, transformers, peft, bitsandbytes
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, 'peft', peft.__version__, 'bnb', bitsandbytes.__version__)


## 4. Hugging Face token

Add `HF_TOKEN` to Colab secrets (key icon in the left sidebar) with write access, then run this cell.
Without it the run still trains; it only skips the Hub push, with a warning. See
`notebooks/README.md` for the full secrets list (`GITHUB_TOKEN`, `HF_TOKEN`).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets')
except Exception as exc:
    print('no HF_TOKEN:', exc)
    print('training will run and checkpoints stay local')


## 5. Data

`data/raw/` and `data/processed/` are gitignored, so the split is regenerated here from the pinned
dataset revision.

`DATA_MODE = 'v1'` runs `data/prepare.py` exactly as v1 did (reject-only cleaning); the audit-only
strict check then proves the hashes match `data/splits.json`, the v1 evidence.

`DATA_MODE = 'v2'` runs the substitute-placeholder cleaning
(`data/prepare.py --placeholder-mode substitute --out data/v2`), writing `data/v2/audit.json` and
`data/v2/splits.json`, with processed rows under `data/processed/v2/`. Verified locally on the Mac
(CPU/mps, node V2-C0, 2026-09-07): `wrote train=20926 val=2753 test=3085 groups=4877`, `strict:
hashes and intersections match`, and a smoke train run against `data/processed/v2` with `--device
mps` completed with `resume match ok=True max_abs_diff=0.00052`. Not yet run on Colab/CUDA; report
the real numbers here once it is.

In [ ]:
DATA_MODE = 'v2'  # 'v1' = v1 as shipped; 'v2' = substitute-placeholder cleaning (V2-B1, landed)

!python data/fetch.py

if DATA_MODE == 'v1':
    !python data/prepare.py
    !python data/prepare.py --audit-only --strict
    DATA_DIR = 'data/processed'
elif DATA_MODE == 'v2':
    !python data/prepare.py --placeholder-mode substitute --out data/v2
    DATA_DIR = 'data/processed/v2'
else:
    raise ValueError(f'unknown DATA_MODE: {DATA_MODE!r}')

print('DATA_DIR =', DATA_DIR)

## 6. Configuration

`CONFIG` picks the hyperparameter file; both are frozen (`docs/dag/CONTRACTS.md`, working rules for
v2) and differ in one line, the row cap. Default here is the free-T4 config, because that is the
config the v1 served checkpoint (`checkpoint-400`) was trained on and the one this run is meant to
reproduce as the v2 headline. Switch to the full split only on an A100 or L4 runtime (check the
`nvidia-smi` output above): a full epoch on the free T4 with `train.yaml` will not finish in a single
Colab session.

In [ ]:
import yaml
from pathlib import Path

CONFIG = 'configs/train-t4.yaml'   # free-T4 budget, 8,000-row cap; swap to 'configs/train.yaml' on A100/L4

cfg = yaml.safe_load(Path(CONFIG).read_text())
RUN_NAME = cfg['run_name']
RUN_DIR = Path('train/runs') / RUN_NAME
SMOKE_DIR = Path('train/runs') / f'smoke-{RUN_NAME}'
print('CONFIG   ', CONFIG)
print('RUN_NAME ', RUN_NAME)
print('RUN_DIR  ', RUN_DIR)
print('DATA_DIR ', DATA_DIR)

## 7. Smoke run: 20 steps, then a resume from step 10

This trains 20 steps on 64 rows drawn from `$DATA_DIR`, drops the later checkpoint, resumes from
step 10 and checks that steps 11 to 20 reproduce the original losses. `smoke.json` records the
comparison; `check_run.py` fails if it did not match. Do not start the full run until this passes.

**If this fails:** an OOM here means the full run will also OOM; drop back to `configs/train-t4.yaml`
or check no other process is holding GPU memory.

In [ ]:
!python train/train.py --config $CONFIG --smoke --data-dir $DATA_DIR
!python tools/check_run.py $SMOKE_DIR --max-memory-gb 12

## 8. Full run

One epoch over `$DATA_DIR`, checkpoints at 50 percent and 100 percent plus every 100 steps. If the
session drops, re-run this cell with `--resume`; it continues from the newest checkpoint and replays
the same data order.

Planning estimates from `docs/TRAINING.md`, not measurements: 30 to 60 minutes on an A100 or L4 for
the full 17,701-row split, roughly 1 to 1.5 hours for the 8,000-row T4 cap. Record the real number
from `config.json`.

Headline rule fixed in advance (GoHighLevel-prep thread, 2026-09-07): this Colab run is the v2
headline training substrate if it finishes clean by 04:00 IST Tuesday; otherwise the parallel Mac
`mps` run (`docs/LOCAL_RUN.md`) is, and `docs/RESULTS.md` says which. Scoring itself always runs on
the sealed Mac Ollama path regardless of which run trained the adapter.

**If this fails:** a dropped Colab session mid-run is normal on the free tier; re-run this exact
cell, uncommenting the `--resume` line, rather than starting over.

In [ ]:
!python train/train.py --config $CONFIG --data-dir $DATA_DIR
# If the session dropped mid-run, use this instead:
# !python train/train.py --config $CONFIG --data-dir $DATA_DIR --resume

## 9. Dev answers for every checkpoint

Each checkpoint is merged into `artifacts/merged`, answered against the 54 dev items through the
transformers backend, and written to `train/runs/<run>/dev-<ckpt>-raw.jsonl`. The merged directory is
removed between checkpoints because `tools/merge.py` refuses to overwrite. `eval/dev.jsonl` is frozen
and the same file regardless of `DATA_MODE`.

**If this fails:** `tools/merge.py` needs the pinned base cached; if Colab has to download it, pass
`--allow-download` by editing the subprocess call below, or run the Hugging Face token cell again.

In [ ]:
import shutil, subprocess
import yaml
from pathlib import Path

run_name = yaml.safe_load(Path(CONFIG).read_text())['run_name']
run_dir = Path('train/runs') / run_name
RUN_DIR = str(run_dir)
checkpoints = sorted(
    (int(p.name.removeprefix('checkpoint-')), p)
    for p in run_dir.glob('checkpoint-*') if (p / 'adapter_config.json').is_file()
)
print('checkpoints:', [step for step, _ in checkpoints])

for step, folder in checkpoints:
    shutil.rmtree('artifacts/merged', ignore_errors=True)
    subprocess.run(['python', 'tools/merge.py', '--adapter', str(folder),
                    '--output', 'artifacts/merged'], check=True)
    out = run_dir / f'dev-checkpoint-{step}-raw.jsonl'
    subprocess.run(['python', 'eval/run.py', '--model', 'tuned', '--backend', 'transformers',
                    '--split', 'dev', '--device', 'cuda', '--output', str(out),
                    '--check-complete'], check=True)
    print('wrote', out)
shutil.rmtree('artifacts/merged', ignore_errors=True)


## 10. Curves and the completeness gate

In [ ]:
!python train/plot_curves.py $RUN_DIR
!python tools/check_run.py $RUN_DIR --max-memory-gb 12
!cat $RUN_DIR/config.json | python -c "import json,sys; d=json.load(sys.stdin); print({k: d[k] for k in ('run_name','device','wall_s','peak_memory_gb','final_step','git_sha')})"


## 11. Selected checkpoint: merge and a dev cross-check, plus optional Ollama

Checkpoint selection is a judgement call on dev pass rate from section 9's
`dev-checkpoint-*-raw.jsonl` files, not something this notebook automates (`docs/LOCAL_RUN.md`:
"Checkpoint selection is D3's job and runs on dev pass rate, not on [the validation loss] number").
Set `CHECKPOINT_STEP` below to the step you are selecting; it defaults to the last checkpoint.

The merge-and-eval cross-check re-answers the dev set through the freshly merged fp16 model, as a
sanity check on the merge step itself, not a new number.

The Ollama cell is optional and slow (it builds llama.cpp from source via `tools/convert.sh`) because
the headline scoring stays on the sealed Mac Ollama path (`docs/dag/CONTRACTS.md` section 6), not on
Colab. Skip it if the notebook's only job today is producing the adapter.

**If the Ollama cell fails:** skip it. It is a visible cross-check of the serving path, not the
scoring path; nothing downstream of this notebook depends on it succeeding.

In [ ]:
import shutil, subprocess
from pathlib import Path

CHECKPOINT_STEP = checkpoints[-1][0]  # override with an int step, e.g. 400, from section 9's results
checkpoint_dir = Path(RUN_DIR) / f'checkpoint-{CHECKPOINT_STEP}'  # RUN_DIR may be a str here: section 9's loop re-set it

shutil.rmtree('artifacts/merged', ignore_errors=True)
subprocess.run(['python', 'tools/merge.py', '--adapter', str(checkpoint_dir),
                '--output', 'artifacts/merged'], check=True)
subprocess.run(['python', 'eval/run.py', '--model', 'tuned', '--backend', 'transformers',
                '--split', 'dev', '--device', 'cuda', '--check-complete'], check=True)
print('merged and cross-checked checkpoint', CHECKPOINT_STEP)

In [ ]:
# Optional. Builds a Q8 GGUF from artifacts/merged with the pinned llama.cpp commit
# (configs/versions.json), installs the Ollama Linux binary, serves it, and runs the exact
# curl from docs/dag/CONTRACTS.md section 6. Expect several minutes for the llama.cpp build.
!pip install -q uv
!apt-get -qq install -y cmake > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
ollama_proc = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

!bash tools/convert.sh artifacts/merged artifacts/tuned-q8.gguf
!cd serve && ollama create ghl-support -f Modelfile
!curl --fail-with-body http://localhost:11434/v1/chat/completions -H 'Content-Type: application/json' \
  -d '{"model":"ghl-support","messages":[{"role":"user","content":"I forgot my password and cannot sign in. What should I do?"}],"temperature":0,"max_tokens":256,"stream":false}'

## 12. Download, and the Hub push that already happened

If `HF_TOKEN` was set in section 4 and `hub.push_checkpoints: true` in the config (it is, in both
frozen configs), `train/train.py` already pushed every checkpoint to `hub.repo_id` as it trained
(section 8's log lines say so). That is the primary way the adapter gets back to the machine that
scores it; `tools/fetch_artifacts.py` there pulls published GGUF artifacts by manifest, a separate,
later step. The zip below is the fallback when the push was skipped or the operator wants the whole
run directory, including `loss.csv` and `smoke.json`, in one file.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(f'/content/{run_dir.name}', 'zip', run_dir)
print('zipped', archive)
files.download(archive)


## What to paste back into the thread

- `DATA_MODE` and `CONFIG` used.
- From `config.json`: `wall_s`, `peak_memory_gb`, `final_step`, `git_sha`, `versions`.
- `smoke.json`: `ok` and `max_abs_diff`.
- The last few lines of `loss.csv`.
- The checkpoint list and the `dev-checkpoint-*-raw.jsonl` row counts.
- `CHECKPOINT_STEP` selected and the merge/dev cross-check result.
- The zip, or the Hub repo URL if the push worked.
- Whether this run or the parallel Mac run is the v2 headline, per the 04:00 IST Tuesday rule.